In [ ]:
from dbrepo.RestClient import RestClient
from dotenv import load_dotenv
import os 
from dbrepo.api.dto import CreateView
from dbrepo.api.dto import CreateView, Subset, SubsetColumn, Join
from dbrepo.api.dto import JoinType

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

containers = client.get_containers()
print(containers)

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [3]:
DB_ID = os.getenv("DB_ID") #"cf27a11d-58e5-4693-856c-e8f3527e3394"

In [4]:
df = client.get_database(DB_ID)
db_tables = client.get_tables(DB_ID)

In [5]:
tab_name_to_id = dict()
for i in db_tables:
    tab_name_to_id[i.name] = i.id

In [6]:
tab_name_to_id

{'gdp_data': 'a7926c82-377b-44f7-819a-b055dff0b5e4',
 'wastewater_data': 'ffb118dc-d4ff-4c86-b79f-49ae73879ea6',
 'city_map': 'e3a73373-d42a-43a3-8c7b-d061910df79f'}

In [7]:
tab_id_and_col_to_col_id = dict()

for tab_id in tab_name_to_id.values():
    table = client.get_table(DB_ID, tab_id)
    for col in table.columns:
        tab_id_and_col_to_col_id[(tab_id, col.name)] = col.id

In [8]:
table = client.get_table(DB_ID, tab_name_to_id["city_map"])
for col in table.columns:
    print(col)

id='c597a750-c2c3-4526-8279-0c74ba77f819' name='nuts_code' database_id='f781c37d-1464-4d86-9791-dc478f8681f2' table_id='e3a73373-d42a-43a3-8c7b-d061910df79f' ord=0 internal_name='nuts_code' is_null_allowed=False type=<ColumnType.VARCHAR: 'varchar'> alias=None description=None size=5 d=None mean=None median=None concept=None unit=None concept_uri='http://purl.org/linked-data/sdmx/2009/dimension#refArea' unit_uri='None' enums=[] sets=[] index_length=None length=None data_length=None max_data_length=None num_rows=None val_min=None val_max=None std_dev=None
id='9ff66509-bcc8-48fb-ba00-d054e74e9001' name='city_name' database_id='f781c37d-1464-4d86-9791-dc478f8681f2' table_id='e3a73373-d42a-43a3-8c7b-d061910df79f' ord=1 internal_name='city_name' is_null_allowed=False type=<ColumnType.VARCHAR: 'varchar'> alias=None description=None size=100 d=None mean=None median=None concept=None unit=None concept_uri='http://purl.obolibrary.org/obo/NCIT_C95378' unit_uri='None' enums=[] sets=[] index_length

# Create View 1: Summary of drugs in water

In [9]:
wtable_id = tab_name_to_id["wastewater_data"]
df_city_summary_view = CreateView(
    name="ww_city_year_drug_summary",
    description="City-level aggregated wastewater indicators per year",
    is_public=True,
    is_schema_public=True,
    query=Subset(
        datasource_ids=[wtable_id],  # wastewater_data table ID

        columns=[
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "city_name")],
                alias="city_name"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "ref_year")],
                alias="ref_year"
            ),

            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "daily_mean_concentration")],
                aggregation="avg",
                alias="avg_daily_mean"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "daily_mean_concentration")],
                aggregation="max",
                alias="max_daily_mean"
            ),

            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "metabolite_name")],
                aggregation="count_distinct",
                alias="metabolite_count"
            )
        ],

        joins=None,
        filters=None,
        orders=None
    )
)


In [10]:
response = client._wrapper(
    method="post",
    url=f"/api/v1/database/{DB_ID}/view",
    payload=df_city_summary_view
)

print(response.status_code)
print(response.text)

201
{"id":"b9e13409-dad3-41ef-b363-7f250cee6dc7","name":"ww_city_year_drug_summary","query":"select `dast_g20_wastewater_epidemiology_kbpc`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_kbpc`.`wastewater_data`.`daily_mean_concentration` as `avg_daily_mean`, `dast_g20_wastewater_epidemiology_kbpc`.`wastewater_data`.`metabolite_name` as `metabolite_count`, `dast_g20_wastewater_epidemiology_kbpc`.`wastewater_data`.`ref_year` as `ref_year` from `wastewater_data`","database_id":"f781c37d-1464-4d86-9791-dc478f8681f2","internal_name":"ww_city_year_drug_summary","is_public":true,"is_schema_public":true,"initial_view":false,"query_hash":"817c76c3a964aadc1b658f3835ebca5ac3ec07c8ae4e1c1e39f6f2c68a069dfd","owned_by":"data_stewardship_group20"}


# Create View 2: Join all

In [11]:
table = client.get_table(DB_ID, tab_name_to_id["city_map"])
print(table)  # or the UUID of the table
for col in table.columns:
    print(col.id, col.name)

id='e3a73373-d42a-43a3-8c7b-d061910df79f' database_id='f781c37d-1464-4d86-9791-dc478f8681f2' name='city_map' owner=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) columns=[Column(id='c597a750-c2c3-4526-8279-0c74ba77f819', name='nuts_code', database_id='f781c37d-1464-4d86-9791-dc478f8681f2', table_id='e3a73373-d42a-43a3-8c7b-d061910df79f', ord=0, internal_name='nuts_code', is_null_allowed=False, type=<ColumnType.VARCHAR: 'varchar'>, alias=None, description=None, size=5, d=None, mean=None, median=None, concept=None, unit=None, concept_uri='http://purl.org/linked-data/sdmx/2009/dimension#refArea', unit_uri='None', enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=None), Column(id='9ff66509-bcc8-48fb-ba00-d054e74e9001', name='city_name', database_id='f781c37d-1464-4d86-9791-dc478f8681f2', table_id='e3a73373-d42a-4

In [12]:
table = client.get_table(DB_ID, tab_name_to_id["gdp_data"])
print(table) 
for col in table.columns:
    print(col.id, col.name)

id='a7926c82-377b-44f7-819a-b055dff0b5e4' database_id='f781c37d-1464-4d86-9791-dc478f8681f2' name='gdp_data' owner=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) columns=[Column(id='868c60ac-ccf4-4c81-944b-fa20ca119833', name='nuts_code', database_id='f781c37d-1464-4d86-9791-dc478f8681f2', table_id='a7926c82-377b-44f7-819a-b055dff0b5e4', ord=0, internal_name='nuts_code', is_null_allowed=False, type=<ColumnType.VARCHAR: 'varchar'>, alias=None, description='5-character NUTS-3 administrative code (e.g. AT221): https://ec.europa.eu/eurostat/web/nuts', size=5, d=None, mean=None, median=None, concept=None, unit=None, concept_uri=None, unit_uri=None, enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=None), Column(id='3d41938b-8e73-45b2-8c49-e7107916b63d', name='ref_year', database_id='f781c37d-1464-4d86-9791-dc478f8

In [13]:
df_ml_view = CreateView(
    name="drug_gdp_features_view",
    description="ML-ready dataset joining wastewater measurements with GDP via city-NUTS mapping",
    is_public=True,
    is_schema_public=True,
    query=Subset(
        datasource_ids=[
            wtable_id  # wastewater_data
        ],

        columns=[
            # wastewater_data
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "city_name")],  # city_name
                alias="city_name"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "ref_year")],  # ref_year
                alias="ref_year"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "metabolite_name")],  # metabolite_name
                alias="metabolite_name"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "daily_mean_concentration")],  # daily_mean_concentration
                alias="daily_mean"
            ),

            # city_map.nuts_code
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(tab_name_to_id["city_map"], "nuts_code")],  # nuts_code in city_map
                alias="nuts_code",
                #join_alias="m"
            ),

            # gdp_data.gdp
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(tab_name_to_id["gdp_data"], "gdp")],  
                alias="gdp",
                #join_alias="g"
            )
        ],

        joins=[
            # wastewater_data → city_map
            Join(
                type=JoinType.INNER,
                datasource_id=tab_name_to_id["city_map"],
                #alias="m",
                conditionals=[
                    {
                        "column_id": tab_id_and_col_to_col_id[(wtable_id, "city_name")],        # wastewater.city_name
                        "foreign_column_id": tab_id_and_col_to_col_id[(tab_name_to_id["city_map"], "city_name")] # city_map.city_name
                    }
                ]
            ),

            # city_map → gdp_data
            Join(
                type=JoinType.INNER,
                datasource_id=tab_name_to_id["gdp_data"],
                #alias="g",
                conditionals=[
                    {
                        "foreign_column_id": tab_id_and_col_to_col_id[(tab_name_to_id["city_map"], "nuts_code")],        # city_map.nuts_code
                        "column_id": tab_id_and_col_to_col_id[(tab_name_to_id["gdp_data"], "nuts_code")] # gdp.nuts_code
                    }
                ]
            )
                    ],

        filters=None,
        orders=None
    )
)



In [14]:
response = client._wrapper(
    method="post",
    url=f"/api/v1/database/{DB_ID}/view",
    payload=df_ml_view
)

print(response.status_code)
print(response.text)

201
{"id":"ece794ef-e22d-4071-a2d5-678806bf7e21","name":"drug_gdp_features_view","query":"select `dast_g20_wastewater_epidemiology_kbpc`.`city_map`.`nuts_code` as `nuts_code`, `dast_g20_wastewater_epidemiology_kbpc`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_kbpc`.`wastewater_data`.`metabolite_name` as `metabolite_name`, `dast_g20_wastewater_epidemiology_kbpc`.`wastewater_data`.`daily_mean_concentration` as `daily_mean`, `dast_g20_wastewater_epidemiology_kbpc`.`gdp_data`.`gdp` as `gdp`, `dast_g20_wastewater_epidemiology_kbpc`.`wastewater_data`.`ref_year` as `ref_year` from `wastewater_data` join `city_map` on `dast_g20_wastewater_epidemiology_kbpc`.`wastewater_data`.`city_name` = `dast_g20_wastewater_epidemiology_kbpc`.`city_map`.`city_name` join `gdp_data` on `dast_g20_wastewater_epidemiology_kbpc`.`gdp_data`.`nuts_code` = `dast_g20_wastewater_epidemiology_kbpc`.`city_map`.`nuts_code`","database_id":"f781c37d-1464-4d86-9791-dc478f8681f2","internal_